In [1]:
!pip install fair-esm
!pip install rdkit
!pip install biopython
!pip install torch torchvision torchaudio --quiet
!pip install torch_geometric --quiet
!pip install scikit-learn matplotlib pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 21.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.5 MB/s eta 0:00:0000:01


In [2]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import pickle

from Bio.PDB import PDBParser, PPBuilder
from rdkit import Chem
from rdkit.Chem import AllChem
from scipy.spatial.distance import cdist

from esm import pretrained
from torch_geometric.data import Data
import esm

In [3]:
AFF_CSV = "/kaggle/input/datasets/sravyasm/mailpdbdataset/pdbbind_v2020_refined_affinity.csv"
REFINED_DIR = "/kaggle/input/datasets/sravyasm/mailpdbdataset/PDBbind_v2020_refined/PDBbind_v2020_refined"
OUT_DIR = "/kaggle/working/processed"
GRAPH_DIR = os.path.join(OUT_DIR, "graphs")
skipped_ligands = []
skipped_empty_pocket = []


os.makedirs(GRAPH_DIR, exist_ok=True)
affinity_df = pd.read_csv(AFF_CSV)
print("Complexes:", len(affinity_df))


Complexes: 18690


In [4]:
from rdkit import Chem
import torch

def load_ligand(mol2_path):
    mol = None

    # Attempt 1: standard permissive load
    try:
        mol = Chem.MolFromMol2File(
            mol2_path,
            sanitize=False,
            removeHs=False
        )
    except Exception:
        mol = None

    # Attempt 2: raw block fallback
    if mol is None:
        try:
            with open(mol2_path, "r") as f:
                mol2_block = f.read()
            mol = Chem.MolFromMol2Block(
                mol2_block,
                sanitize=False,
                removeHs=False
            )
        except Exception:
            mol = None

    # Final fail → return None (do NOT raise)
    if mol is None or mol.GetNumAtoms() == 0:
        return None

    # Use crystal coordinates
    conf = mol.GetConformer()
    coords = conf.GetPositions()

    atom_feats = []
    for atom in mol.GetAtoms():
        atom_feats.append([
            atom.GetAtomicNum(),
            atom.GetFormalCharge(),
            int(atom.GetIsAromatic()),
            int(atom.GetHybridization())
        ])

    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in mol.GetBonds()]

    return (
        torch.tensor(atom_feats, dtype=torch.float),
        torch.tensor(coords, dtype=torch.float),
        bonds
    )


In [5]:
def load_protein_residues(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_path)

    residues = []
    coords = []

    for model in structure:
        for chain in model:
            for res in chain:
                if res.id[0] != " ":
                    continue
                if "CA" not in res:
                    continue
                residues.append(res)
                coords.append(res["CA"].get_coord())
        break

    return residues, torch.tensor(coords, dtype=torch.float)


In [6]:
def get_pocket_residues(lig_coords, prot_coords, cutoff=8.0):
    d = cdist(lig_coords.cpu(), prot_coords.cpu())
    mask = d.min(axis=0) <= cutoff
    idx = np.where(mask)[0]

    if len(idx) == 0:
        idx = np.argsort(d.min(axis=0))[:200]

    return idx


In [7]:
def extract_sequence(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_path)
    ppb = PPBuilder()

    seq = ""
    for pp in ppb.build_peptides(structure):
        seq += str(pp.get_sequence())
    return seq if seq else None
    
sequence_cache = {}
for _, row in tqdm(affinity_df.iterrows(), desc="Extracting sequences"):
    cid = row["complex_id"]
    pdb = os.path.join(REFINED_DIR, cid, f"{cid}_protein.pdb")
    if os.path.exists(pdb):
        seq = extract_sequence(pdb)
        if seq:
            sequence_cache[cid] = seq

with open(os.path.join(OUT_DIR, "protein_sequences.pkl"), "wb") as f:
    pickle.dump(sequence_cache, f)


Extracting sequences: 0it [00:00, ?it/s]

In [8]:
model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model.eval().cuda()
batch_converter = alphabet.get_batch_converter()
MAX_LEN = 1024

seq = seq[:MAX_LEN]

batch = [("seq", seq)]
_, _, tokens = batch_converter(batch)
tokens = tokens.cuda()

with torch.no_grad():
    out = model(tokens, repr_layers=[12])

reps = out["representations"][12][0, 1:len(seq)+1].cpu()

del tokens, out
torch.cuda.empty_cache()

print(reps.shape)  # (seq_len, 480)


def batch_esm(seqs):
    embeddings = []

    for seq in seqs:
        if len(seq) > 1024:
            seq = seq[:1024]

        batch = [("seq", seq)]
        _, _, tokens = batch_converter(batch)
        tokens = tokens.cuda()

        with torch.no_grad():
            out = model(tokens, repr_layers=[12])

        rep = out["representations"][12][0, 1:len(seq)+1].cpu()
        embeddings.append(rep)

        del tokens, out
        torch.cuda.empty_cache()

    return embeddings

esm_cache = {}
cids = list(sequence_cache.keys())
seqs = [sequence_cache[c] for c in cids]
embs = batch_esm(seqs)

esm_cache = dict(zip(cids, embs))


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D-contact-regression.pt
torch.Size([945, 480])


In [9]:
import torch
import numpy as np
from torch_geometric.data import Data
from scipy.spatial.distance import cdist

def build_graph(
    lig_x, lig_pos, lig_bonds,
    prot_residues, prot_pos, pocket_idx,
    esm_embedding, y,
    cutoff=8.0
):
    esm_dim = 480

    # --- Clip pocket indices safely ---
    max_esm_len = esm_embedding.size(0)
    pocket_idx = pocket_idx[pocket_idx < max_esm_len]
    if len(pocket_idx) == 0:
        return None

    # --- Center coordinates (optional but recommended) ---
    center = lig_pos.mean(dim=0)
    lig_pos = lig_pos - center
    prot_pos = prot_pos - center

    # --- Ligand node features ---
    lig_esm = torch.zeros(lig_x.size(0), esm_dim)
    lig_full = torch.cat([lig_x, lig_esm], dim=1)

    # --- Protein node features ---
    prot_struct = torch.zeros(len(pocket_idx), lig_x.size(1))
    prot_esm = esm_embedding[pocket_idx]
    prot_full = torch.cat([prot_struct, prot_esm], dim=1)

    # --- Combine nodes ---
    x = torch.cat([lig_full, prot_full], dim=0)

    # --- Add node-type encoding (0 = ligand, 1 = protein) ---
    L = lig_x.size(0)
    P = len(pocket_idx)

    node_type = torch.cat([
        torch.zeros(L, 1),
        torch.ones(P, 1)
    ], dim=0)

    x = torch.cat([x, node_type], dim=1)

    # --- Build edges ---
    src, dst = [], []

    # 1️⃣ Ligand–Ligand (bond edges)
    for i, j in lig_bonds:
        src += [i, j]
        dst += [j, i]

    # 2️⃣ Protein–Protein (distance-based)
    prot_coords = prot_pos[pocket_idx]
    d_pp = cdist(prot_coords, prot_coords)
    i_pp, j_pp = np.where((d_pp < cutoff) & (d_pp > 0))

    src += (i_pp + L).tolist()
    dst += (j_pp + L).tolist()

    # 3️⃣ Ligand–Protein interaction edges
    d_lp = cdist(lig_pos, prot_coords)
    i_lp, j_lp = np.where(d_lp < cutoff)

    src += i_lp.tolist()
    dst += (j_lp + L).tolist()

    src += (j_lp + L).tolist()
    dst += i_lp.tolist()

    # --- Final edge tensor ---
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    # --- Combine positions ---
    pos = torch.cat([lig_pos, prot_coords], dim=0)

    return Data(
        x=x,
        edge_index=edge_index,
        pos=pos,
        y=torch.tensor([y], dtype=torch.float)
    )

In [10]:
for _, row in tqdm(affinity_df.iterrows(), desc="Building graphs"):
    cid = row["complex_id"]
    y = row["pK"]

    lig_path = os.path.join(REFINED_DIR, cid, f"{cid}_ligand.mol2")
    prot_path = os.path.join(REFINED_DIR, cid, f"{cid}_protein.pdb")

    if not (os.path.exists(lig_path) and os.path.exists(prot_path)):
        continue
    if cid not in esm_cache:
        continue

    lig_data = load_ligand(lig_path)
    if lig_data is None:
        skipped_ligands.append(cid)
        continue
    
    lig_x, lig_pos, lig_bonds = lig_data

    prot_res, prot_pos = load_protein_residues(prot_path)

    pocket_idx = get_pocket_residues(lig_pos, prot_pos)

    data = build_graph(
        lig_x, lig_pos, lig_bonds,
        prot_res, prot_pos,
        pocket_idx,
        esm_cache[cid],
        y
    )
    if data is None:
        skipped_empty_pocket.append(cid)
        continue
    torch.save(data, os.path.join(GRAPH_DIR, f"{cid}.pt"))


Building graphs: 0it [00:00, ?it/s]

/tmp/ipykernel_55/1086695430.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  return residues, torch.tensor(coords, dtype=torch.float)
[18:58:45] 1epq_ligand: warning - O.co2 with non C.2 or S.o2 neighbor.
[18:58:45] 1epq_ligand: warning - O.co2 with non C.2 or S.o2 neighbor.
[18:59:28] 1bq4_ligand: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[19:02:28] 1nki_ligand: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[19:03:36] 1t5a_ligand: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[19:04:35] 2hwg_ligand: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[19:04:42] 2dua_ligand: Warning - no explicit hydrogens in 

In [11]:
import os

GRAPH_DIR = "/kaggle/working/processed/graphs"

pt_files = [f for f in os.listdir(GRAPH_DIR) if f.endswith(".pt")]
print("Number of .pt files:", len(pt_files))


Number of .pt files: 18649


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import matplotlib.pyplot as plt
from torch_geometric.nn import GCNConv, SAGEConv, GINConv, GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
import os
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.calibration import calibration_curve
from scipy.stats import spearmanr

In [13]:
import os
import torch
import numpy as np
import pickle
from torch_geometric.loader import DataLoader

# ============================
# 1️⃣ Load graphs and extract IDs
# ============================

graph_files = [f for f in os.listdir(GRAPH_DIR) if f.endswith(".pt")]

graphs = []
complex_ids = []

for f in graph_files:
    cid = f.replace(".pt", "")  # PDB ID
    data = torch.load(os.path.join(GRAPH_DIR, f), weights_only=False)
    graphs.append(data)
    complex_ids.append(cid)

print("Total graphs loaded:", len(graphs))


# ============================
# 2️⃣ Protein-Level Split
# ============================

SPLIT_SEED = 42
split_path = "/kaggle/working/protein_split.pkl"

if os.path.exists(split_path):
    print("Loading existing protein-level split...")
    with open(split_path, "rb") as f:
        split_dict = pickle.load(f)

    train_ids = split_dict["train_ids"]
    val_ids = split_dict["val_ids"]
    test_ids = split_dict["test_ids"]

else:
    print("Creating new protein-level split...")

    np.random.seed(SPLIT_SEED)

    unique_ids = np.array(complex_ids)
    np.random.shuffle(unique_ids)

    n = len(unique_ids)
    n_train = int(0.7 * n)
    n_val = int(0.15 * n)

    train_ids = unique_ids[:n_train]
    val_ids = unique_ids[n_train:n_train+n_val]
    test_ids = unique_ids[n_train+n_val:]

    split_dict = {
        "train_ids": train_ids,
        "val_ids": val_ids,
        "test_ids": test_ids
    }

    with open(split_path, "wb") as f:
        pickle.dump(split_dict, f)

print("Train proteins:", len(train_ids))
print("Val proteins:", len(val_ids))
print("Test proteins:", len(test_ids))


# ============================
# 3️⃣ Assign graphs to splits
# ============================

train_graphs = []
val_graphs = []
test_graphs = []

train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

for graph, cid in zip(graphs, complex_ids):
    if cid in train_set:
        train_graphs.append(graph)
    elif cid in val_set:
        val_graphs.append(graph)
    elif cid in test_set:
        test_graphs.append(graph)

print("Train:", len(train_graphs))
print("Val:", len(val_graphs))
print("Test:", len(test_graphs))


# ============================
# 4️⃣ Create DataLoaders
# ============================

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=32)
test_loader = DataLoader(test_graphs, batch_size=32)

print("DataLoaders ready.")

Total graphs loaded: 18649
Creating new protein-level split...
Train proteins: 13054
Val proteins: 2797
Test proteins: 2798
Train: 13054
Val: 2797
Test: 2798
DataLoaders ready.


In [14]:


class GAT3(nn.Module):
    def __init__(self, in_dim=4, hidden_dim=128, heads=4, dropout=0.2):
        super().__init__()

        self.conv1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout)
        self.conv3 = GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout)

        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim * heads)
        self.bn3 = nn.BatchNorm1d(hidden_dim * heads)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * heads, 1)

    def forward(self, x, edge_index, batch):
        
        x = F.elu(self.conv1(x, edge_index))
        x = self.bn1(x)
        x = self.dropout(x)

        x = F.elu(self.conv2(x, edge_index))
        x = self.bn2(x)
        x = self.dropout(x)

        x = F.elu(self.conv3(x, edge_index))
        x = self.bn3(x)
        x = self.dropout(x)

        x = global_mean_pool(x, batch)
        return self.fc(x).squeeze(-1)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            preds.append(out.cpu().numpy())
            trues.append(batch.y.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    mse = mean_squared_error(trues, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(trues, preds)
    return mse, rmse, r2, preds, trues

def mc_evaluate(model, loader, device, n_passes=30):
    model.train()  # Enable dropout for uncertainty
    all_mean_preds = []
    all_std_preds = []
    all_trues = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            preds = []
            for _ in range(n_passes):
                out = model(batch.x, batch.edge_index, batch.batch)
                preds.append(out.cpu().numpy())
            preds = np.stack(preds)  # [n_passes, batch_size]
            mean_pred = preds.mean(axis=0)
            std_pred = preds.std(axis=0)
            all_mean_preds.append(mean_pred)
            all_std_preds.append(std_pred)
            all_trues.append(batch.y.cpu().numpy())
    all_mean_preds = np.concatenate(all_mean_preds)
    all_std_preds = np.concatenate(all_std_preds)
    all_trues = np.concatenate(all_trues)
    return all_mean_preds, all_std_preds, all_trues

def compute_ece(preds, trues, n_bins=10):
    bin_bounds = np.linspace(min(preds), max(preds), n_bins + 1)
    bin_indices = np.digitize(preds, bin_bounds) - 1
    ece = 0.0
    for i in range(n_bins):
        in_bin = bin_indices == i
        if np.any(in_bin):
            bin_preds = preds[in_bin]
            bin_trues = trues[in_bin]
            bin_error = mean_absolute_error(bin_trues, bin_preds)
            bin_weight = len(bin_trues) / len(trues)
            ece += bin_weight * bin_error
    return ece



In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
example_in_dim = graphs[0].x.shape[1]

EPOCHS = 50
seeds = [0, 1, 2, 3, 4]
seed_rmses = []
seed_r2s = []
seed_corrs = []
seed_eces = []

for seed in seeds:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    model = GAT3(
        in_dim=example_in_dim,
        hidden_dim=128,
        heads=4,           # ← add this (or whatever you want)
        dropout=0.2
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5
    )
    criterion = nn.MSELoss()

    best_val_rmse = float("inf")
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        val_mse, val_rmse, val_r2, _, _ = evaluate(
            model, val_loader, device
        )
        print(
            f"Seed {seed} | Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val RMSE: {val_rmse:.4f} | "
            f"R2: {val_r2:.4f}"
        )
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            torch.save(model.state_dict(), f"GAT3_seed{seed}.pt")
            print(f"✓ Saved best model for seed {seed} (Val RMSE={val_rmse:.4f})")

    model.load_state_dict(torch.load(f"GAT3_seed{seed}.pt"))
    test_mse, test_rmse, test_r2, det_preds, det_trues = evaluate(model, test_loader, device)
    seed_rmses.append(test_rmse)
    seed_r2s.append(test_r2)

    # MC for uncertainty corr
    mean_preds, std_preds, trues = mc_evaluate(model, test_loader, device)
    abs_errors = np.abs(mean_preds - trues)
    corr, _ = spearmanr(std_preds, abs_errors)
    seed_corrs.append(corr)

    # ECE on deterministic preds
    ece = compute_ece(det_preds, det_trues)
    seed_eces.append(ece)

    print(f"Seed {seed} TEST: RMSE={test_rmse:.4f}, R2={test_r2:.4f}, Unc-Err Corr={corr:.4f}, ECE={ece:.4f}")

print("\nAGGREGATE RESULTS (5 seeds):")
print(f"RMSE mean: {np.mean(seed_rmses):.4f}, range: {min(seed_rmses):.4f}-{max(seed_rmses):.4f}")
print(f"R2 mean: {np.mean(seed_r2s):.4f}, range: {min(seed_r2s):.4f}-{max(seed_r2s):.4f}")
print(f"Unc-Err Corr mean: {np.mean(seed_corrs):.4f}")
print(f"ECE mean: {np.mean(seed_eces):.4f}")

Using device: cuda
Seed 0 | Epoch 001 | Train Loss: 9.3119 | Val RMSE: 1.7754 | R2: 0.0057
✓ Saved best model for seed 0 (Val RMSE=1.7754)
Seed 0 | Epoch 002 | Train Loss: 2.5634 | Val RMSE: 1.7221 | R2: 0.0645
✓ Saved best model for seed 0 (Val RMSE=1.7221)
Seed 0 | Epoch 003 | Train Loss: 2.3781 | Val RMSE: 1.7466 | R2: 0.0377
Seed 0 | Epoch 004 | Train Loss: 2.3971 | Val RMSE: 1.6722 | R2: 0.1180
✓ Saved best model for seed 0 (Val RMSE=1.6722)
Seed 0 | Epoch 005 | Train Loss: 2.2678 | Val RMSE: 1.6591 | R2: 0.1317
✓ Saved best model for seed 0 (Val RMSE=1.6591)
Seed 0 | Epoch 006 | Train Loss: 2.2621 | Val RMSE: 1.6211 | R2: 0.1710
✓ Saved best model for seed 0 (Val RMSE=1.6211)
Seed 0 | Epoch 007 | Train Loss: 2.2080 | Val RMSE: 1.5485 | R2: 0.2436
✓ Saved best model for seed 0 (Val RMSE=1.5485)
Seed 0 | Epoch 008 | Train Loss: 2.1612 | Val RMSE: 1.6258 | R2: 0.1662
Seed 0 | Epoch 009 | Train Loss: 2.1357 | Val RMSE: 1.8963 | R2: -0.1343
Seed 0 | Epoch 010 | Train Loss: 2.1087 | Va